# 04 — ModelOpt Q/DQ ONNX Export
Change `QUANT_DTYPE` to `'int8'`, `'fp8'`, or `'int4'` and run all.

In [18]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('../src').resolve()))

In [19]:
import numpy as np
from data import get_dataloader
from config import ExperimentConfig
import modelopt.onnx.quantization as moq

In [20]:
from model import get_model
from onnx_exporter import ONNXExporter

model = get_model(cfg=None, pretrained=False)

exporter = ONNXExporter(
    model=model,
    device="cpu",
    onnx_path="../onnx/resnet18.onnx",
)
exporter.export_model(opset_version=17, dynamic_batch=True)

/home/pf4636/code/resnet/quantized_resnets/src/onnx_exporter.py:28: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0404 10:38:26.369000 968171 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0404 10:38:26.512000 968171 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 

[onnx_exporter] Exporting to ../onnx/resnet18.onnx ...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
[onnx_exporter] Saved (0.1 MB)


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


PosixPath('../onnx/resnet18.onnx')

In [21]:
QUANT_DTYPE    = "int8"        #fp8, int4, int8  
ONNX_IN        = "../onnx/resnet18.onnx"
ONNX_OUT       = f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx"
CALIB_BATCHES  = 32

In [22]:
# Build calibration numpy array from dataloader
cfg = ExperimentConfig(device="cpu", batch_size=32)
loader = get_dataloader(cfg, split="val")

batches = []
for i, (images, _) in enumerate(loader):
    if i >= CALIB_BATCHES:
        break
    batches.append(images.numpy())

calib_data = np.concatenate(batches, axis=0)
print(f"Calibration data: {calib_data.shape}")

Calibration data: (1024, 3, 224, 224)


moq.quantize(
        onnx_path=ONNX_IN,
        quantize_mode=QUANT_DTYPE,
        calibration_data=calib_data,
        output_path=ONNX_OUT,
        op_types_to_quantize=["Conv", "MatMul", "Gemm", "Add"]
        
    )
print(f"Saved → {ONNX_OUT}")

In [23]:
import onnx as _onnx
_FP8_OPS = {"Conv", "Gemm", "MatMul", "Add"}
_nodes_to_q = None
if QUANT_DTYPE == "fp8":
    _m = _onnx.load(ONNX_IN)
    _nodes_to_q = [n.name for n in _m.graph.node if n.op_type in _FP8_OPS]
    print(f"Explicit nodes_to_quantize: {len(_nodes_to_q)} nodes "
          f"({sum(1 for n in _m.graph.node if n.op_type == 'Add')} Add nodes included)")

moq.quantize(
    onnx_path=ONNX_IN,
    quantize_mode=QUANT_DTYPE,
    calibration_data=calib_data,
    output_path=ONNX_OUT,
    op_types_to_quantize=list(_FP8_OPS) if QUANT_DTYPE == "fp8" else None,
    nodes_to_quantize=_nodes_to_q,
)
print(f"Saved → {ONNX_OUT}")

[modelopt][onnx] - INFO - Starting quantization process for model: ../onnx/resnet18.onnx
[modelopt][onnx] - INFO - Quantization mode: int8
[modelopt][onnx] - INFO - Preprocessing the model ../onnx/resnet18.onnx
[modelopt][onnx] - INFO - Model has dynamic inputs: ['images']
[modelopt][onnx] - INFO - Found 0 custom layers and 59 tensors
[modelopt][onnx] - INFO - No custom ops found. If that's not correct, please make sure that the 'tensorrt' python package is correctly installed and that the paths to 'libcudnn*.so' and TensorRT 'lib/' are in 'LD_LIBRARY_PATH'. If the custom op is not directly available as a plugin in TensorRT, please also make sure that the path to the compiled '.so' TensorRT plugin is also being given via the  '--trt_plugins' flag (requires TRT 10+).
[modelopt][onnx] - INFO - Duplicating shared constants
[modelopt][onnx] - INFO - Setting up CalibrationDataProvider for calibration
[modelopt][onnx] - INFO - Analyzing MHA nodes for int8 quantization
[modelopt][onnx] - INFO

100%|██████████| 46/46 [00:00<00:00, 1870.77it/s]


100%|██████████| 46/46 [00:00<00:00, 1810.15it/s]


100%|██████████| 46/46 [00:00<00:00, 1621.26it/s]


100%|██████████| 46/46 [00:00<00:00, 1779.79it/s]


...

100%|██████████| 46/46 [00:00<00:00, 1646.48it/s]


100%|██████████| 46/46 [00:00<00:00, 1642.65it/s]


100%|██████████| 46/46 [00:00<00:00, 1739.78it/s]


100%|██████████| 46/46 [00:00<00:00, 1751.75it/s]


Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 46
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
[modelopt][onnx] - INFO - Starting post-processing of quantized model
[modelopt][onnx] - INFO - Deleting QDQ nodes from marked inputs to make certain operations fusible
[modelopt][onnx] - INFO - Converting float32 tensors to fp16
[modelopt][onnx] - INFO - Quantization completed successfully in 38.40323352813721 seconds
[modelopt][onnx] - INFO - Total number of nodes: 135
[modelopt][onnx] - INFO - Total number of quantized nodes: 30
[modelopt][onnx] - INFO - Quantized onnx model is saved as ../onnx/resnet18_int8_qdq.onnx
[modelopt][onnx] - INFO - Cleaning up intermediate files
[modelopt][onnx] - INFO - Validating quantized model
[modelopt][onnx] - INFO - Quantization process completed
Saved → ../onnx/resnet18_int8_qdq.onnx


In [24]:
import onnx
ops = [n.op_type for n in onnx.load(f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx").graph.node]
print(f"Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")

Q: 41, DQ: 41


For Int4:

This is the key insight you're missing: INT4 almost always means weight-only quantization, which is a fundamentally different scheme.
Here's why. INT4 has only 16 discrete values (or 8 for unsigned). That's far too coarse for activations, which vary dynamically at inference time. So instead of the full Q/DQ pattern, INT4 tools do this:

Weights are quantized offline and stored as INT4 constants directly in the graph.
A single DequantizeLinear converts them back to float at runtime, right before the matmul/conv.
Activations remain in float — no Q node is needed for them.

So the graph doesn't look like → Q → DQ → Op →. Instead it looks like:
INT4 weight (constant) → DQ → MatMul(activation, dequantized_weight)
There's no Q node because the quantization happened statically when the model was exported — the weights are already stored as INT4. You only need DQ to convert them back to float for computation. The single DQ: 1 you see likely means only one layer was eligible (or the tool only quantized one layer of ResNet18 to INT4).

ResNet18 has ~20 Conv layers. Each gets one Q/DQ for weights and one for activations, giving ~40 pairs. You get 38 instead of 40 because a few layers are typically excluded from quantization (e.g., the first conv or final FC layer, which are sensitive to precision loss).

Compare: int8 gives Q:41/DQ:41 — the small difference just reflects which layers each mode's calibration decided to quantize.

In [25]:
ops = [n.op_type for n in onnx.load(f"../onnx/resnet18_int8_qdq.onnx").graph.node]
print(f"Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")

Q: 41, DQ: 41


INT4 — 0 Q, 1 DQ (weight-only quantization)

Weights are pre-quantized to INT4 statically at export time and stored as INT4 constants in the graph. No Q node is needed at runtime — the quantization already happened. Only a DQ is inserted to convert INT4 weights → float just before the compute op. Activations stay in float entirely. The "1 DQ" means only 1 layer was eligible (INT4 is too coarse for most of ResNet18's small layers).

INT8 — 41 Q, 41 DQ (full activation + weight quantization)

Both weights and activations are quantized dynamically. Every quantizable op gets a Q/DQ pair on inputs: ~20 Conv layers × 2 tensors (weight + activation) ≈ 40, plus the final FC layer = 41.

FP8 — 38 Q, 38 DQ (3 fewer than INT8)

Same scheme as INT8 (dynamic Q/DQ for both weights and activations), but 3 fewer pairs. This is because modelopt's FP8 mode excludes certain layers that INT8 quantizes:

The first Conv layer is commonly excluded — input pixel distributions don't fit FP8's narrow range well
Residual addition ops or other element-wise ops may be excluded since FP8 arithmetic isn't supported for them in TensorRT
The final FC/classifier may also be excluded for accuracy reasons
INT8 is more permissive about which layers it wraps; FP8 is stricter because the format (E4M3 or E5M2) has a much smaller representable range than INT8.

In [26]:
import onnx
from onnx import TensorProto

model = onnx.load("/home/pf4636/code/resnet/quantized_resnets/onnx/resnet18_fp8_qdq.onnx")

dtype_map = {v: k for k, v in TensorProto.DataType.items()}

for init in model.graph.initializer:
    dtype = dtype_map.get(init.data_type, str(init.data_type))
    print(f"{init.name[:60]:<60} {dtype}")

layer1.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer1.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer1.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer1.1.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer2.0.downsample.0.weight_zero_point_1                    FLOAT8E4M3FN
layer2.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer2.1.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.conv2.weight_zero_point_1                           FLOAT8E4M3FN
layer3.0.downsample.0.weight_zero_point_1                    FLOAT8E4M3FN
layer3.1.conv1.weight_zero_point_1                           FLOAT8E4M3FN
layer3.1.conv2.weight_zero_point_1    